In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Data folder:", RAW_PATH)

Project root: c:\Users\hp\Projects\retail-intelligence-analytics
Data folder: c:\Users\hp\Projects\retail-intelligence-analytics\data\raw


In [3]:
expected_files = [
    "transactions.csv",
    "products.csv",
    "supplier_costs.csv",
    "price_history.csv",
    "inventory.csv",
    "returns.csv",
    "customers.csv",
    "marketing_spend.csv"
]

missing_files = [
    name for name in expected_files
    if not (RAW_PATH / name).exists()
]

assert not missing_files, f"Missing files: {missing_files}"

print("All eight datasets found.")

All eight datasets found.


In [4]:
datasets = {}

for filename in expected_files:
    name = Path(filename).stem

    datasets[name] = pd.read_csv(
        RAW_PATH / filename
    )

    print(f"Loaded {name}: {datasets[name].shape}")

Loaded transactions: (100000, 16)
Loaded products: (500, 9)
Loaded supplier_costs: (998, 12)
Loaded price_history: (18000, 12)
Loaded inventory: (500, 7)
Loaded returns: (7126, 8)
Loaded customers: (8000, 11)
Loaded marketing_spend: (216, 13)


In [5]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data types:")
    print(df.dtypes)


===== TRANSACTIONS =====
Shape: (100000, 16)
Columns: ['transaction_id', 'customer_id', 'product_id', 'date', 'quantity', 'unit_price_usd', 'discount_pct', 'revenue_usd', 'cost_usd', 'profit_usd', 'shipping_cost_usd', 'channel', 'payment_method', 'status', 'country', 'category']
Data types:
transaction_id           str
customer_id              str
product_id               str
date                     str
quantity               int64
unit_price_usd       float64
discount_pct         float64
revenue_usd          float64
cost_usd             float64
profit_usd           float64
shipping_cost_usd    float64
channel                  str
payment_method           str
status                   str
country                  str
category                 str
dtype: object

===== PRODUCTS =====
Shape: (500, 9)
Columns: ['product_id', 'name', 'category', 'brand', 'unit_price_usd', 'unit_cost_usd', 'weight_kg', 'is_active', 'launch_date']
Data types:
product_id            str
name                  st

In [6]:
inventory_records = []

for name, df in datasets.items():
    inventory_records.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

inventory_df = pd.DataFrame(inventory_records)

display(inventory_df)

,dataset,rows,columns,missing_cells,duplicate_rows
0,transactions,100000,16,0,0
1,products,500,9,0,0
2,supplier_costs,998,12,0,0
3,price_history,18000,12,0,0
4,inventory,500,7,0,0
5,returns,7126,8,0,0
6,customers,8000,11,0,0
7,marketing_spend,216,13,0,0


In [7]:
for name, df in datasets.items():
    missing = df.isna().sum()

    print(f"\n{name}:")
    print(missing[missing > 0])


transactions:
Series([], dtype: int64)

products:
Series([], dtype: int64)

supplier_costs:
Series([], dtype: int64)

price_history:
Series([], dtype: int64)

inventory:
Series([], dtype: int64)

returns:
Series([], dtype: int64)

customers:
Series([], dtype: int64)

marketing_spend:
Series([], dtype: int64)


In [8]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print(name, "duplicate rows:", duplicate_count)

transactions duplicate rows: 0
products duplicate rows: 0
supplier_costs duplicate rows: 0
price_history duplicate rows: 0
inventory duplicate rows: 0
returns duplicate rows: 0
customers duplicate rows: 0
marketing_spend duplicate rows: 0


In [9]:
key_candidates = {
    "transactions": "transaction_id",
    "products": "product_id",
    "customers": "customer_id"
}

for name, key in key_candidates.items():
    df = datasets[name]

    if key in df.columns:
        print(
            name,
            "unique:", df[key].is_unique,
            "missing:", df[key].isna().sum()
        )
    else:
        print(name, "does not contain", key)

transactions unique: True missing: 0
products unique: True missing: 0
customers unique: True missing: 0


In [10]:
transactions = datasets["transactions"]
products = datasets["products"]
customers = datasets["customers"]

missing_products = ~transactions["product_id"].isin(
    products["product_id"]
)

missing_customers = ~transactions["customer_id"].isin(
    customers["customer_id"]
)

print(
    "Transactions with unknown products:",
    missing_products.sum()
)

print(
    "Transactions with unknown customers:",
    missing_customers.sum()
)

Transactions with unknown products: 0
Transactions with unknown customers: 0


In [11]:
financial_columns = [
    "quantity",
    "unit_price_usd",
    "discount_pct",
    "revenue_usd",
    "cost_usd",
    "profit_usd"
]

for column in financial_columns:
    if column in transactions.columns:
        print(
            column,
            "Minimum:", transactions[column].min(),
            "Maximum:", transactions[column].max()
        )

quantity Minimum: 1 Maximum: 10
unit_price_usd Minimum: 8.06 Maximum: 1191.59
discount_pct Minimum: 0.0 Maximum: 0.25
revenue_usd Minimum: 6.45 Maximum: 11915.9
cost_usd Minimum: 4.58 Maximum: 8936.9
profit_usd Minimum: -0.03 Maximum: 2979.0


In [12]:
EXPORT_PATH = PROJECT_ROOT / "data" / "exports"

EXPORT_PATH.mkdir(parents=True, exist_ok=True)

inventory_df.to_csv(
    EXPORT_PATH / "dataset_inventory.csv",
    index=False
)

print("Dataset inventory exported.")

Dataset inventory exported.
